In [1]:
import os
import torch
import clip
import faiss
import numpy as np
from PIL import Image

# Set device to GPU if available, otherwise CPU.
device = "cuda" if torch.cuda.is_available() else "cpu"
# device = "cpu"

# Load the CLIP model and the corresponding preprocessing function.
model, preprocess = clip.load("ViT-B/32", device=device)

def load_images(image_folder):
    """
    Loads and preprocesses images from the specified folder.
    
    Args:
        image_folder (str): Path to the folder containing images.
        
    Returns:
        list: A list of tuples where each tuple contains (image_path, preprocessed_image_tensor).
    """
    valid_extensions = (".png", ".jpg", ".jpeg")
    image_paths = [os.path.join(image_folder, file)
                   for file in os.listdir(image_folder)
                   if file.lower().endswith(valid_extensions)]
    
    images = []
    for path in image_paths:
        try:
            image = Image.open(path).convert("RGB")
            image_tensor = preprocess(image).unsqueeze(0).to(device)
            images.append((path, image_tensor))
        except Exception as e:
            print(f"Error processing {path}: {e}")
    return images

def compute_image_embeddings(images):
    """
    Computes normalized image embeddings using CLIP.
    
    Args:
        images (list): List of tuples (image_path, image_tensor).
        
    Returns:
        np.ndarray: Array of image embeddings.
        list: Corresponding list of image paths.
    """
    embeddings = []
    image_paths = []
    with torch.no_grad():
        for path, image_tensor in images:
            # Encode image to get its feature vector.
            image_features = model.encode_image(image_tensor)
            # Normalize the embedding vector.
            image_features = image_features / image_features.norm(dim=-1, keepdim=True)
            embeddings.append(image_features.cpu().numpy())
            image_paths.append(path)
    embeddings = np.vstack(embeddings)
    return embeddings, image_paths

def build_faiss_index(embeddings):
    """
    Builds a FAISS index using inner product (IP) which, with normalized embeddings, 
    is equivalent to cosine similarity.
    
    Args:
        embeddings (np.ndarray): Array of image embeddings.
    
    Returns:
        faiss.Index: A FAISS index built on the embeddings.
    """
    dimension = embeddings.shape[1]
    index = faiss.IndexFlatIP(dimension)
    index.add(embeddings)
    return index

def search_index(query, index, image_paths, k=1):
    """
    Searches the FAISS index using the text query's embedding.
    
    Args:
        query (str): User's text query.
        index (faiss.Index): FAISS index containing image embeddings.
        image_paths (list): List of image paths corresponding to the embeddings.
        k (int): Number of nearest neighbors to retrieve.
    
    Returns:
        tuple: (distances, indices) from the FAISS search.
    """
    with torch.no_grad():
        # Tokenize and encode the text query.
        text_input = clip.tokenize([query]).to(device)
        text_features = model.encode_text(text_input)
        # Normalize the text feature vector.
        text_features = text_features / text_features.norm(dim=-1, keepdim=True)
        text_embedding = text_features.cpu().numpy()
    
    distances, indices = index.search(text_embedding, k)
    return distances, indices



In [ ]:
# Update with the path to your folder containing images.
image_folder = r"C:\Users\Research-Lab\Desktop\FateZeroEdit\FateZero\data\negative_reg\car"

# Load and compute embeddings for all images.
images = load_images(image_folder)
if not images:
    print("No images found in the specified folder.")
    exit(1)

embeddings, image_paths = compute_image_embeddings(images)
# Build the FAISS index from the image embeddings.
index = build_faiss_index(embeddings)

# Get the text query from the user.
query = "red porsche car" # input("Give the prompt: ")
distances, indices = search_index(query, index, image_paths, k=1)

best_idx = indices[0][0]
best_image_path = image_paths[best_idx]
similarity = distances[0][0]

print(f"Most similar image: {best_image_path}")
print(f"Similarity score: {similarity:.4f}")

In [4]:
# !pip install openai-clip 

  Using cached openai-clip-1.0.1.tar.gz (1.4 MB)
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Using cached ftfy-6.3.1-py3-none-any.whl.metadata (7.3 kB)
Using cached ftfy-6.3.1-py3-none-any.whl (44 kB)
  Created wheel for openai-clip: filename=openai_clip-1.0.1-py3-none-any.whl size=1368609 sha256=4ce97e43fe83d06fc4f788500cca9b2618f9deae7d16afd55e026c48ed6a7200
  Stored in directory: c:\users\research-lab\appdata\local\pip\cache\wheels\ab\49\bc\c2342e8e14878210ba4825cf314a53f2570f6fb18b91fce3cf
Successfully built openai-clip
